# Outlook

In this notebook we code a simple version of the DQN algorithm — **without**
a replay buffer nor a target network — so as to better understand its inner
mechanisms. The next notebook adds those two ingredients.

The DQN algorithm is explained in [this
video](https://www.youtube.com/watch?v=CXwvOMJujZk) and you can also read
[the corresponding slides](http://pages.isir.upmc.fr/~sigaud/teach/dqn.pdf).

## Setup

**On Colab**: run the installation cell below (generated automatically,
pinned versions). **Locally**: install the environment with `uv sync`, no
need for this cell.

In [1]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, replace
from torch import Tensor
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from rl_mind.core import Action, Actor
from rl_mind.nn import build_mlp
from rl_mind.env import VecEnv
from rl_mind.data import Episode
from rl_mind.collectors import EpisodeCollector
from rl_mind.evaluation import Evaluator
from rl_mind.notebook import run_directory, setup_tensorboard, silence_known_warnings


## The `rl_mind` toolkit

The `rl_mind` package ([documentation](https://pypi.org/project/rl-mind/)),
installed with the practicals, provides a small set of *typed* helpers used
throughout the RL notebooks. The ones used here:

- `VecEnv` runs several copies of a gymnasium environment in parallel;
- `Episode` stores one full episode (`obs` `[T, obs_dim]`, `action` `[T]`,
  `reward` `[T]`, `final_obs`, `terminated`), plus `all_obs` `[T+1, obs_dim]`
  — the observations $s_0, \ldots, s_T$, i.e. `obs` with `final_obs`
  appended;
- `EpisodeCollector` runs an actor in a `VecEnv` and returns complete
  `Episode`s;
- `Evaluator` periodically evaluates an actor and keeps the best one;
- `Actor` is the base class of the policies: `actor(obs)` is the
  training-time (exploratory) behavior, `actor.act(obs)` the deterministic
  (evaluation) one.

In [2]:
# pygame and tensorboard still use the deprecated `pkg_resources` API
silence_known_warnings()

# DQN

[DQN](https://daiwk.github.io/assets/dqn.pdf) is a **critic-only** algorithm
for discrete actions: a single network $Q(s, a)$ both estimates the action
values and, through its argmax, defines the policy.

## Definition of the agents

### The Q-network

The Q-network takes a state and outputs one value per action, $Q(s, \cdot)$
(so its output layer has `n_actions` neurons).

In [3]:
class DiscreteQNetwork(nn.Module):
    """The action-value network $Q(s, \\cdot)$ for discrete actions"""

    def __init__(self, obs_dim: int, hidden: tuple[int, ...], n_actions: int):
        super().__init__()
        self.model = build_mlp([obs_dim, *hidden, n_actions])

    def forward(self, obs: Tensor) -> Tensor:
        """Q-values for a batch of observations: `[B, obs_dim] -> [B, n_actions]`"""
        return self.model(obs)

### The policy: $\epsilon$-greedy exploration

The policy is derived from the Q-network. Like Q-learning, DQN needs some
exploration to avoid converging too early, so at training time we use
**$\epsilon$-greedy**: with probability $\epsilon$ a random action, otherwise
the greedy action $\arg\max_a Q(s, a)$. At evaluation time (`act`) we always
take the greedy action.

`QPolicy` is an `Actor`: it *wraps* the Q-network and turns its values into
actions, so it can be handed to the collector and the evaluator directly.

In [4]:
class QPolicy(Actor[Action]):
    """An $\\epsilon$-greedy policy over a `DiscreteQNetwork`"""

    def __init__(self, q_network: DiscreteQNetwork, epsilon: float):
        super().__init__()
        self.q_network = q_network
        self.epsilon = epsilon

    def forward(self, obs: Tensor) -> Action:
        q_values = self.q_network(obs)
        greedy = q_values.argmax(-1)
        random = torch.randint(q_values.shape[-1], (obs.shape[0],))
        explore = torch.rand(obs.shape[0]) < self.epsilon
        return Action(value=torch.where(explore, random, greedy))

    def act(self, obs: Tensor) -> Tensor:
        return self.q_network(obs).argmax(-1)

## The critic loss

The `compute_critic_loss` function implements the Bellman backup. In
Q-learning the update rule reads

$$Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha\,[\, r_{t+1} + \gamma \max_a
Q(s_{t+1},a) - Q(s_t,a_t)\,]$$

In DQN the $Q \leftarrow Q + \alpha[\delta]$ step is replaced by a gradient
step minimizing the squared temporal difference: we regress $Q(s_t, a_t)$
towards the target $r_{t+1} + \gamma \max_a Q(s_{t+1}, a)$.

Three points to get right:

- the target uses $\max_a Q(s_{t+1}, a)$, **detached** — we do not want the
  gradient to flow through the target, only through $Q(s_t, a_t)$;
- $Q(s_t, a_t)$ picks, among the `n_actions` outputs, the one for the action
  that was actually played — that's what
  [`torch.gather`](https://pytorch.org/docs/stable/generated/torch.gather.html)
  (or fancy indexing) does;
- the successors $s_1, \ldots, s_T$ of the observations in `episode.obs` are
  `episode.all_obs[1:]` — the last one, $s_T$, is `episode.final_obs`, which
  is not in `episode.obs` since no action was taken there;
- the value of that last successor must be dropped from the target when the
  episode **terminated** (no future) — but kept when it was merely
  *truncated* (a time limit: the agent could have continued).

In [21]:
def compute_critic_loss(
    gamma: float, episode: Episode, q_network: DiscreteQNetwork
) -> Tensor:
    """Compute the DQN temporal-difference loss over one episode

    :param gamma: The discount factor
    :param episode: The episode (tensors of length `T`)
    :param q_network: The Q-network (used for both $Q(s_t,a_t)$ and, detached,
        the bootstrap $\\max_a Q(s_{t+1},a)$ — there is no target network yet)
    :return: The critic loss (a scalar)
    """
    # Compute the loss

    # Q(s_t, a_t) for the actions that were played (gradient flows here)
    q_values = q_network(episode.obs)
    q_sa = torch.gather(
        input=q_values,
        dim=1,
        index=episode.action.value.unsqueeze(-1)
    ).squeeze(-1)
    # max_a Q(s_{t+1}, a), detached; the successors of `obs` are `all_obs[1:]`
    with torch.no_grad():
        q_successors = q_network(episode.all_obs[1:])
        q_max = q_successors.max(dim=1).values
        terminated = torch.zeros(size=(len(episode.obs),), dtype=torch.bool)
        terminated[-1] = episode.terminated
        target = episode.reward + gamma * ~terminated  * q_max

    # bootstrap is 0 for the last step iff the episode terminated
    return ((target - q_sa) ** 2).mean()


## Hyper-parameters

In [6]:
@dataclass(frozen=True)
class DQNConfig:
    env_name: str = "CartPole-v1"
    seed: int = 3

    #: Number of gradient updates
    max_updates: int = 700
    #: Number of parallel training environments
    n_envs: int = 8
    #: Episodes collected per update
    episodes_per_update: int = 8

    #: Exploration rate
    epsilon: float = 0.1
    #: Discount factor
    gamma: float = 0.99
    max_grad_norm: float = 0.5

    hidden: tuple[int, ...] = (256, 256)
    lr: float = 2e-3

    #: Steps between two evaluations, and number of evaluation episodes
    eval_interval: int = 5_000
    n_eval_envs: int = 10

## Main training loop

The environments run **without** a replay buffer: each update collects a
fresh batch of full episodes with the current $\epsilon$-greedy policy,
averages the per-episode critic loss and takes one gradient step.

In [7]:
def run_dqn(cfg: DQNConfig) -> Evaluator:
    """Train a DQN agent (no replay buffer, no target network)

    :param cfg: The hyper-parameters
    :return: The evaluator, which holds the evaluation history and the best
        policy found (`visualize_best()` shows it playing)
    """
    # Fix the random seed so that two runs with the same `cfg` give the same
    # result (network initialization, exploration, ...)
    torch.manual_seed(cfg.seed)

    # --- Setup

    # `n_envs` copies of the environment, stepped together: the collector
    # gathers episodes from all of them at once
    env = VecEnv(cfg.env_name, cfg.n_envs, seed=cfg.seed)

    # The Q-network: observation -> one Q-value per action
    q_network = DiscreteQNetwork(env.observation_dim, cfg.hidden, env.n_actions)

    # The epsilon-greedy policy *wraps* the Q-network (it does not copy it):
    # each time the network is updated, the policy changes too
    policy = QPolicy(q_network, cfg.epsilon)

    # The optimizer updates the parameters of the Q-network (the only network
    # we train here) from the gradient of the loss
    optimizer = torch.optim.Adam(q_network.parameters(), lr=cfg.lr)

    # The collector runs `policy` (training behavior, i.e. with exploration)
    # in `env`, and returns full episodes
    collector = EpisodeCollector(env, policy)

    # Logging: a directory for this run, and a tensorboard writer in it (the
    # curves appear in the tensorboard panel launched below)
    run_dir = run_directory(f"dqn-simple-{cfg.env_name}-S{cfg.seed}")
    writer = SummaryWriter(run_dir)

    # The evaluator measures the *greedy* policy (`policy.act`, no
    # exploration) every `eval_interval` steps, on its own environments (a
    # different seed: we do not evaluate on the training episodes), and keeps
    # a copy of the best policy so far
    evaluator = Evaluator(
        VecEnv(cfg.env_name, cfg.n_eval_envs, seed=cfg.seed + 100),
        every=cfg.eval_interval,
        run_dir=run_dir,
        writer=writer,
    )

    # --- Training loop: one gradient update per iteration

    pbar = tqdm(range(cfg.max_updates))  # a progress bar
    for update in pbar:
        # 1. Play `episodes_per_update` full episodes with the current policy
        episodes = collector.collect(cfg.episodes_per_update)

        # 2. The loss: the mean over the episodes of the per-episode TD loss
        critic_loss = torch.stack(
            [compute_critic_loss(cfg.gamma, ep, q_network) for ep in episodes]
        ).mean()

        # 3. One gradient step: reset the gradients (PyTorch accumulates them
        # otherwise), back-propagate, clip the gradient norm (large gradients
        # make DQN unstable), and update the parameters
        optimizer.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(q_network.parameters(), cfg.max_grad_norm)
        optimizer.step()

        # 4. Log the loss, and evaluate if `eval_interval` steps have passed
        # since the last evaluation (`result` is `None` otherwise). The x-axis
        # is the number of environment steps, not of updates, so that
        # algorithms that do not update at the same pace can be compared
        writer.add_scalar("loss/critic", critic_loss.item(), collector.steps)
        if result := evaluator.run_if_needed(collector.steps, policy):
            pbar.set_description(
                f"eval={result.mean:6.1f} best={evaluator.best_reward:6.1f}"
            )

    return evaluator

## Launching tensorboard to visualize the results

In [12]:
%reload_ext tensorboard

In [9]:
setup_tensorboard()

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [22]:
cfg = DQNConfig()
dqn = run_dqn(cfg)

  0%|          | 0/700 [00:00<?, ?it/s]

In [16]:
dqn.visualize_best()

## What's next?

This simple version has no replay buffer and no target network — it is
unstable and sample-inefficient. The next notebook (`03-2`) adds:

- a **replay buffer** (learning from a stream of transitions rather than one
  fresh batch of episodes);
- a **target network** for the bootstrap (stabilizing the regression
  target);
- and, as an exercise, **Double-DQN** to fight the overestimation bias.